# Baby Qwen 2/3: Fine-Tuning (SFT) to Follow Instructions

**Target Architecture**: Scaled-down Qwen2.5 (~42M parameters)  
**Curriculum Goals**:
- **Step 1: Pretraining from Scratch** (`week4/BabyQwen-1-Pretraining.ipynb`) — Causal language modeling on raw text to build a base model (*raw text → language*).
- **Step 2: Supervised Fine-Tuning** (`week5/BabyQwen-2-FineTuning.ipynb`) — ChatML template & response-only loss to teach instruction-following (*language → follows instructions*).
- **Step 3: Knowledge Injection & LoRA** (`week5/BabyQwen-3-KnowledgeInjection.ipynb`) — Parameter-efficient fine-tuning to inject new domain facts into the tuned model (*follows instructions → knows new facts*).

### Step 2 vs Step 3 at a glance

| | **This notebook, Step 2: BabyQwen-2 (SFT)** | **Step 3: BabyQwen-3 (Knowledge Injection)** |
|---|---|---|
| **Teaches** | *How* to answer: the chat format and doing what an instruction asks | *What* to answer: specific facts the model never saw |
| **Starts from** | `baby_qwen_pretrained` (only continues text) | `baby_qwen_sft` (already follows instructions) |
| **Data** | about 4,000 instruction/response pairs built from ordinary stories | 48 facts about an invented world (Willowbrook), asked in several phrasings |
| **Method** | full fine-tuning and LoRA, compared side by side | LoRA only: a small adapter on a frozen model |
| **Checked by** | does the answer contain the requested word or name? | recall of seen, rephrased and never-seen facts, plus forgetting |
| **Saves** | `baby_qwen_sft` (full model) | adapter (about 1 MB) and a merged copy |

Both steps train on prompt → response pairs with the loss on the response tokens only. Step 2 uses that recipe to teach the model **how to answer**; Step 3 reuses it, through a LoRA adapter, to teach it **what to answer**.

**Current Focus**: **Step 2** — The pretrained model from Step 1 is a *story continuer*. Give it text and it keeps writing similar text. It has never seen a **request** and a **response**. Supervised fine-tuning (SFT) teaches exactly that:

> prompt in a chat template → response, with the loss computed **only on the response tokens**.

## Overview

1. **Chat template + loss masking**: the core mechanics of SFT.
2. **Measuring instruction-following** with an automatic metric, not by eyeballing samples.
3. **Full fine-tuning vs LoRA** on the same data: quality, trainable parameters, time.
4. **Optional preview of preference tuning (DPO)** on the SFT model, with the caveat that results are noisy at this scale (section 10).
5. **Why model size matters**: a 42M model can learn a *format* and a *behavior*, but SFT on stories gives it no new *facts*. Adding facts is the subject of Step 3 (`BabyQwen-3`), which continues from the `baby_qwen_sft` checkpoint saved here.

## The task

Three kinds of instructions, all built from TinyStories text the model has never seen:

| Type | Instruction | Response | Automatic check |
|---|---|---|---|
| `words` | Write a short story using the word: *kite*. | a story | the word appears |
| `name` | Write a short story about a character named *Mia*. | a story | the name appears |
| `continue` | Continue this story: *first two sentences* | the rest of the story | (none; judged by loss) |

| Specification | Value |
|---|---|
| Starting point | `week4/baby_qwen_pretrained` (from Step 1) |
| SFT data | about 4,000 instruction/response pairs from unseen stories |
| Held-out data | stories from the TinyStories `validation` split |
| Methods compared | Base (no SFT), LoRA (r=16), Full fine-tuning |
| Destination | `./baby_qwen_sft` & `./baby_qwen_lora_adapter` (passed to Step 3) |
| Optional preview | DPO on the SFT model, off by default, saved to `./baby_qwen_dpo` (not used by Step 3) |

In [ ]:
import subprocess, sys
try:
    import peft
except ImportError:   # e.g. a fresh Colab runtime
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'peft'])

import os, re, math, time, random
from collections import Counter
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
random.seed(0)
torch.manual_seed(0)
print(f'PyTorch {torch.__version__} | device: {device.upper()}')

---
## 1. Load the Pretrained Checkpoint from Week 4

In [ ]:
# Output folder of BabyQwen-1. On Colab, upload it next to this notebook (or into /content) first.
CANDIDATES = ['../week4/baby_qwen_pretrained', 'baby_qwen_pretrained', '/content/baby_qwen_pretrained']
BASE_DIR = next((p for p in CANDIDATES if os.path.isdir(p)), None)
assert BASE_DIR, f'baby_qwen_pretrained not found in {CANDIDATES}. Run week4/BabyQwen-1-Pretraining.ipynb first.'

tokenizer = AutoTokenizer.from_pretrained(BASE_DIR)
IM_START = tokenizer.convert_tokens_to_ids('<|im_start|>')
IM_END = tokenizer.convert_tokens_to_ids('<|im_end|>')
EOT = tokenizer.convert_tokens_to_ids('<|endoftext|>')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_base():
    return AutoModelForCausalLM.from_pretrained(BASE_DIR, dtype=torch.float32).to(device)

model = load_base()
n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded Baby Qwen2.5: {n_params:,} parameters')
print(f'Special token ids: <|im_start|>={IM_START}  <|im_end|>={IM_END}  <|endoftext|>={EOT}')

> **A subtle point.** `<|im_start|>` and `<|im_end|>` exist in the Qwen vocabulary, but Baby Qwen never saw them in pretraining. Its stories were delimited with `<|endoftext|>`. The embeddings for the chat tokens are therefore still at their random initialization. A real pretrained Qwen has learned them. SFT must teach Baby Qwen both what the template *means* and when to emit `<|im_end|>` to stop.

---
## 2. The Chat Template and Loss Masking

Every example is rendered in ChatML:

```
<|im_start|>system
You are a friendly storyteller.<|im_end|>
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>
```

The model sees the whole sequence, but the loss is $\mathcal{L} = -\frac{1}{|R|}\sum_{t \in R} \log P(x_t \mid x_{<t})$ over **response tokens $R$ only**. We don't train the model to predict the system prompt or the user's instruction. That text is *given* at inference time. Masked positions get label `-100`, which PyTorch's cross-entropy ignores.

In [ ]:
SYSTEM = 'You are a friendly storyteller.'

def chat_prompt(instruction):
    return (f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
            f'<|im_start|>user\n{instruction}<|im_end|>\n'
            f'<|im_start|>assistant\n')

def encode_example(ex):
    # Returns token ids and labels. Prompt tokens get label -100 (ignored by the loss).
    prompt_ids = tokenizer.encode(chat_prompt(ex['instr']))
    resp_ids = tokenizer.encode(ex['resp']) + [IM_END]
    return prompt_ids + resp_ids, [-100] * len(prompt_ids) + resp_ids

# Show the masking on a tiny example
demo = {'instr': 'Write a short story about a character named Mia.', 'resp': 'Mia had a red ball.'}
ids, labels = encode_example(demo)
print(f"{'token':<16} {'trained on?'}")
for i, l in list(zip(ids, labels))[-9:]:
    print(f'{tokenizer.decode([i])!r:<16} {"YES" if l != -100 else "no (masked)"}')

---
## 3. Build the Instruction Dataset

We build instructions *programmatically* from raw TinyStories text so that the right answer is checkable by code. Stories for SFT come **after** the first 6,000 train stories, which BabyQwen-1 already used for pretraining. Evaluation stories come from the separate `validation` split.

> **Use your own data.** To fine-tune on your own instruction/response pairs, set `CUSTOM_PAIRS` at the top of the next cell, for example `[{'instr': 'Summarize: ...', 'resp': '...'}, ...]` or `'my_pairs.jsonl'`. You need at least 20 pairs, and each pair must fit in `MAX_LEN` tokens. Keep expectations modest: with a 42M-parameter model this teaches *format and style* more than skill.


In [ ]:
# ── USE YOUR OWN DATA (optional) ───────────────────────────────────────────
# CUSTOM_PAIRS: a list of {'instr': ..., 'resp': ...} dicts, or the path to a .jsonl file with
#               one such object per line. Leave as None to use the TinyStories tasks below.
# The ChatML template, system prompt and response-only loss are unchanged. A custom run has no
# automatic word/name check, so watch the held-out response loss and read the generated samples.
CUSTOM_PAIRS = None
CUSTOM_VAL_FRACTION = 0.1

PRETRAIN_STORIES = 6_000   # used by BabyQwen-1, so we skip them
N_POOL = 12_000            # candidate stories for SFT
N_TRAIN = 4_000
N_VAL_STORIES = 400
MAX_LEN = 192   # short stories only: faster steps, and the instruction is never truncated away

def take(split, skip, n):
    if CUSTOM_PAIRS is not None: return []   # no need to download TinyStories
    ds = load_dataset('roneneldan/TinyStories', split=split, streaming=True).skip(skip)
    return [ex['text'].strip() for _, ex in zip(range(n), ds)]

pool = take('train', PRETRAIN_STORIES, N_POOL)
val_stories = take('validation', 0, N_VAL_STORIES)

# Document frequency over the pool: content words are neither too rare nor too common
doc_freq = Counter(w for s in pool for w in set(re.findall(r'[a-z]{4,}', s.lower())))
def is_content_word(w):
    return 10 <= doc_freq[w] <= 0.04 * len(pool)

NOT_NAMES = {'I', 'Mom', 'Dad', 'Mommy', 'Daddy', 'Grandma', 'Grandpa', 'Mama', 'Papa'}
def sentences(story):
    return re.split(r'(?<=[.!?])\s+', story)

def build_words(story, rng):
    cands = sorted(w for w in set(re.findall(r'[a-z]{4,}', story.lower())) if is_content_word(w))
    if len(cands) < 1: return None
    ws = rng.sample(cands, 1)
    return dict(kind='words', instr=f'Write a short story using the word: {ws[0]}.', resp=story, meta=ws)

def build_name(story, rng):
    # Names are capitalized words that appear mid-sentence at least twice
    counts = Counter()
    for sent in sentences(story):
        for tok in sent.split()[1:]:
            tok = tok.strip('.,!?"\'')
            tok = tok[:-2] if tok.endswith("'s") else tok
            if tok.istitle() and tok.isalpha() and tok not in NOT_NAMES:
                counts[tok] += 1
    if not counts or counts.most_common(1)[0][1] < 2: return None
    name = counts.most_common(1)[0][0]
    return dict(kind='name', instr=f'Write a short story about a character named {name}.', resp=story, meta=name)

def build_continue(story, rng):
    s = sentences(story)
    if len(s) < 5: return None
    return dict(kind='continue', instr='Continue this story: ' + ' '.join(s[:2]), resp=' '.join(s[2:]), meta=None)

BUILDERS = [build_words, build_name, build_continue]

def make_examples(stories, n, seed):
    # Rotate through the three types, falling back to the next type if a story doesn't fit
    rng = random.Random(seed)
    out = []
    for i, story in enumerate(stories):
        for k in range(3):
            ex = BUILDERS[(i + k) % 3](story, rng)
            if ex is not None:
                ids, _ = encode_example(ex)
                if len(ids) <= MAX_LEN:
                    out.append(ex)
                break
        if len(out) >= n: break
    return out

train_examples = make_examples(pool, N_TRAIN, seed=1)
val_examples = make_examples(val_stories, 10_000, seed=2)

if CUSTOM_PAIRS is not None:
    if isinstance(CUSTOM_PAIRS, str):
        import json
        CUSTOM_PAIRS = [json.loads(line) for line in open(CUSTOM_PAIRS, encoding='utf-8') if line.strip()]
    bad = [k for k, p in enumerate(CUSTOM_PAIRS)
           if not (isinstance(p, dict) and str(p.get('instr', '')).strip() and str(p.get('resp', '')).strip())]
    assert not bad, f"Pairs at indexes {bad[:5]} need non-empty 'instr' and 'resp' fields."
    pairs = [dict(kind='custom', instr=p['instr'].strip(), resp=p['resp'].strip(), meta=None) for p in CUSTOM_PAIRS]
    usable = [p for p in pairs if len(encode_example(p)[0]) <= MAX_LEN]   # drop examples longer than MAX_LEN tokens
    print(f'{len(usable)} of {len(pairs)} pairs fit within MAX_LEN={MAX_LEN} tokens')
    assert len(usable) >= 20, 'Need at least 20 usable pairs (raise MAX_LEN if your examples are long).'
    random.Random(0).shuffle(usable)
    n_val = max(2, int(CUSTOM_VAL_FRACTION * len(usable)))
    val_examples, train_examples = usable[:n_val], usable[n_val:]
print(f'Train examples: {len(train_examples)} | Val examples: {len(val_examples)}')
print('Train mix:', dict(Counter(e['kind'] for e in train_examples)))
for kind in ['words', 'name', 'continue', 'custom']:
    ex = next((e for e in train_examples if e['kind'] == kind), None)
    if ex:
        print(f"\n[{kind}] {ex['instr'][:110]}\n   -> {ex['resp'][:130]}...")

---
## 4. Training and Evaluation Utilities

We write the SFT loop explicitly (as in BabyQwen-1) so every step is visible: mask → forward → backward → clip → step. The same function trains both the LoRA and the full fine-tuning model.

In [ ]:
def collate(examples):
    # Right-pad a batch; padded label positions are -100 so they are ignored
    enc = [encode_example(e) for e in examples]
    L = max(len(i) for i, _ in enc)
    ids = torch.full((len(enc), L), tokenizer.pad_token_id, dtype=torch.long)
    lab = torch.full((len(enc), L), -100, dtype=torch.long)
    att = torch.zeros((len(enc), L), dtype=torch.long)
    for r, (i, l) in enumerate(enc):
        ids[r, :len(i)] = torch.tensor(i); lab[r, :len(l)] = torch.tensor(l); att[r, :len(i)] = 1
    return ids.to(device), lab.to(device), att.to(device)

@torch.no_grad()
def response_loss(m, examples, bs=8):
    # Token-weighted mean cross-entropy over response tokens only
    m.eval()
    total, count = 0.0, 0
    for i in range(0, len(examples), bs):
        ids, lab, att = collate(examples[i:i + bs])
        logits = m(input_ids=ids, attention_mask=att).logits[:, :-1]
        tgt = lab[:, 1:]
        total += F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), tgt.reshape(-1),
                                 ignore_index=-100, reduction='sum').item()
        count += (tgt != -100).sum().item()
    return total / count

def train(m, data, val_data, epochs, lr, bs=8, warmup_frac=0.05, label=''):
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, betas=(0.9, 0.95), weight_decay=0.0)
    steps_per_epoch = math.ceil(len(data) / bs)
    total_steps = epochs * steps_per_epoch
    warmup = max(1, int(warmup_frac * total_steps))
    def lr_at(s):
        if s < warmup: return lr * (s + 1) / warmup
        p = (s - warmup) / max(1, total_steps - warmup)
        return 0.1 * lr + 0.9 * lr * 0.5 * (1 + math.cos(math.pi * p))

    log = {'step': [], 'loss': [], 'val_step': [0], 'val_loss': [response_loss(m, val_data)]}
    print(f'[{label}] step 0 | val loss {log["val_loss"][0]:.4f}')
    t0, step = time.time(), 0
    for ep in range(epochs):
        order = list(range(len(data))); random.Random(ep).shuffle(order)
        m.train()
        for b in range(0, len(order), bs):
            ids, lab, att = collate([data[j] for j in order[b:b + bs]])
            for g in opt.param_groups: g['lr'] = lr_at(step)
            logits = m(input_ids=ids, attention_mask=att).logits[:, :-1]
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), lab[:, 1:].reshape(-1), ignore_index=-100)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); step += 1
            log['step'].append(step); log['loss'].append(loss.item())
        v = response_loss(m, val_data)
        log['val_step'].append(step); log['val_loss'].append(v)
        print(f'[{label}] epoch {ep + 1}/{epochs} | step {step}/{total_steps} | train loss {log["loss"][-1]:.4f} | val loss {v:.4f} | {time.time() - t0:.0f}s')
    return log, time.time() - t0

@torch.no_grad()
def generate(m, instructions, max_new_tokens=220, bs=20):
    # Batched greedy decoding with left padding; stops at <|im_end|> or <|endoftext|>
    m.eval()
    tokenizer.padding_side = 'left'
    outs = []
    for i in range(0, len(instructions), bs):
        enc = tokenizer([chat_prompt(x) for x in instructions[i:i + bs]], return_tensors='pt', padding=True).to(m.device)
        g = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1,
                       eos_token_id=[IM_END, EOT], pad_token_id=tokenizer.pad_token_id)
        outs += tokenizer.batch_decode(g[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    tokenizer.padding_side = 'right'
    return [o.strip() for o in outs]

def follows(ex, text):
    if ex['kind'] == 'words':
        return all(re.search(rf'\b{w}\b', text.lower()) for w in ex['meta'])
    return re.search(rf'\b{ex["meta"]}\b', text) is not None

N_EVAL = 60
eval_words = [e for e in val_examples if e['kind'] == 'words'][:N_EVAL]
eval_names = [e for e in val_examples if e['kind'] == 'name'][:N_EVAL]
loss_val = val_examples[:150]   # mixed-type held-out set for response loss
DEMO_PROMPTS = [
    'Write a short story using the word: kite.',
    'Write a short story about a character named Mia.',
    'Continue this story: One day, a little fox found a shiny key. He wanted to know what it opened.',
]
if CUSTOM_PAIRS is not None:
    DEMO_PROMPTS = [e['instr'] for e in val_examples[:3]]   # held-out prompts from your data

def evaluate(m, label):
    # Instruction-following rate for the two checkable task types, plus held-out response loss
    res = {'label': label, 'val_loss': response_loss(m, loss_val)}
    for kind, exs in [('words', eval_words), ('name', eval_names)]:
        outs = generate(m, [e['instr'] for e in exs]) if exs else []
        res[kind] = sum(follows(e, o) for e, o in zip(exs, outs)) / len(exs) if exs else float('nan')
    res['demo'] = generate(m, DEMO_PROMPTS)
    print(f"[{label}] val loss {res['val_loss']:.3f} | words {res['words']:.0%} | name {res['name']:.0%}")
    return res

print(f'Eval prompts: {len(eval_words)} words, {len(eval_names)} name | loss set: {len(loss_val)}')

---
## 5. Baseline: The Pretrained Model Before SFT

We run the same checks on the model straight out of BabyQwen-1. It has no notion of an instruction, so we expect low instruction-following and a high response loss. The chat tokens are still random embeddings. This baseline is what SFT must beat.

In [ ]:
results = {}
results['base'] = evaluate(model, 'Base (no SFT)')
print('\nSample generation before SFT:')
print('Prompt   :', DEMO_PROMPTS[1])
print('Generated:', results['base']['demo'][1][:400])

---
## 6. Method A: LoRA SFT

LoRA freezes the 42M pretrained weights and trains small low-rank updates $\Delta W = \frac{\alpha}{r}BA$ on every attention and FFN projection.

Two choices differ from the typical large-model recipe, and both matter at this scale:
- **Higher rank (r=16) and higher learning rate (1e-3).** On a model with $d=256$, rank 16 is a much bigger fraction of $d$ than rank 8 would be on $d=896$, and LoRA generally wants a learning rate 5-10x higher than full fine-tuning.
- **The embedding table is frozen.** The chat-token embeddings stay at their random init. The adapters have to learn to work with them, which is the harder case and a good stress test for LoRA.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
lora_model = get_peft_model(load_base(), lora_config)
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in lora_model.parameters())
print(f'LoRA trainable parameters: {trainable:,} / {total:,} ({trainable / total:.2%})')

EPOCHS = 3
BS = 16
lora_log, lora_time = train(lora_model, train_examples, loss_val, epochs=EPOCHS, lr=1e-3, bs=BS, label='LoRA')
results['lora'] = evaluate(lora_model, 'LoRA SFT')
results['lora'].update(trainable=trainable, time=lora_time)

LORA_DIR = 'baby_qwen_lora_adapter'
lora_model.save_pretrained(LORA_DIR)   # adapter only: a small fraction of the model size
print(f'Saved LoRA adapter to {LORA_DIR}/ ({os.path.getsize(os.path.join(LORA_DIR, "adapter_model.safetensors")) / 1e6:.1f} MB)')

---
## 7. Method B: Full Fine-Tuning

With only 42M parameters, updating everything is cheap. Full fine-tuning can also *move the embedding table*, so the chat tokens get proper embeddings. We use a lower learning rate (5e-4) because every weight moves.

In [ ]:
del lora_model
if device == 'mps': torch.mps.empty_cache()

full_model = load_base()
trainable = sum(p.numel() for p in full_model.parameters() if p.requires_grad)
print(f'Full fine-tuning trainable parameters: {trainable:,} (100%)')

full_log, full_time = train(full_model, train_examples, loss_val, epochs=EPOCHS, lr=5e-4, bs=BS, label='Full FT')
results['full'] = evaluate(full_model, 'Full fine-tuning')
results['full'].update(trainable=trainable, time=full_time)

SFT_DIR = 'baby_qwen_sft'
full_model.save_pretrained(SFT_DIR)
tokenizer.save_pretrained(SFT_DIR)
print(f'Saved full fine-tuned model to {SFT_DIR}/')

---
## 8. Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for log, color, name in [(lora_log, 'steelblue', 'LoRA'), (full_log, 'crimson', 'Full FT')]:
    axes[0].plot(log['step'], log['loss'], color=color, alpha=0.4, lw=1)
    axes[0].plot(log['val_step'], log['val_loss'], color=color, marker='o', lw=2, label=f'{name} (val)')
axes[0].axhline(results['base']['val_loss'], color='gray', ls='--', label='Base model (val)')
axes[0].set_title('Response-only loss (thin = train, thick = held-out)')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss (nats)'); axes[0].grid(alpha=0.3); axes[0].legend()

names = ['Base', 'LoRA', 'Full FT']
keys = ['base', 'lora', 'full']
x = range(3)
axes[1].bar([i - 0.2 for i in x], [results[k]['words'] for k in keys], 0.4, label='words', color='seagreen')
axes[1].bar([i + 0.2 for i in x], [results[k]['name'] for k in keys], 0.4, label='name', color='darkorange')
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(names)
axes[1].set_ylim(0, 1); axes[1].set_title('Instruction-following rate on held-out prompts')
axes[1].grid(alpha=0.3, axis='y'); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"{'Model':<18} {'Trainable':>12} {'Time':>7} {'Val loss':>9} {'Words':>7} {'Name':>7}")
print('-' * 64)
for k, lab in zip(keys, ['Base (no SFT)', 'LoRA r=16', 'Full fine-tune']):
    r = results[k]
    tr = f"{r['trainable']:,}" if 'trainable' in r else '-'
    tm = f"{r['time']:.0f}s" if 'time' in r else '-'
    print(f"{lab:<18} {tr:>12} {tm:>7} {r['val_loss']:>9.3f} {r['words']:>7.0%} {r['name']:>7.0%}")

In [ ]:
for i, prompt in enumerate(DEMO_PROMPTS):
    print('=' * 78)
    print('PROMPT:', prompt)
    for k, lab in zip(keys, ['Base', 'LoRA', 'Full FT']):
        print(f'\n--- {lab} ---')
        print(results[k]['demo'][i][:420])

---
## 9. Reading the Results

- **Format learned**: after SFT the model follows the template and stops cleanly at `<|im_end|>`. Before SFT it doesn't, because the chat tokens carried no meaning.
- **Instruction-following**: compare the `Words` and `Name` columns against the base row. Copying a requested word or name into the story is the behavior SFT can teach a small model, but a 42M model learns it only partially, and more data and epochs help.
- **LoRA vs full fine-tuning**: compare the rows above. With a tiny model and a frozen embedding table, LoRA trains a small fraction of the weights, while full fine-tuning updates everything. On a large model LoRA nearly matches full fine-tuning, but the gap is usually wider at this scale.
- **Limits**: the stories are simple and sometimes illogical. SFT changed *behavior*, not *capability*. The model's knowledge of the world is still whatever 1.2M tokens of pretraining gave it.

### Contrast: a real instruction-tuned model

The same prompts on **Qwen2.5-0.5B-Instruct**, which has about 12x more parameters and was pretrained on trillions of tokens (cached from BabyQwen-3):

In [ ]:
del full_model
if device == 'mps': torch.mps.empty_cache()

if not (eval_words and eval_names):
    print('Skipped: this contrast uses the word/name checks, which are not available for custom data.')
else:
    # Qwen2.5-0.5B-Instruct shares our tokenizer family, so the same ChatML prompts work.
    # The system prompt and chat_prompt() are identical; only the weights differ.
    qwen = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct', dtype=torch.float16).to(device)
    q_res = {'label': 'Qwen2.5-0.5B-Instruct'}
    for kind, exs in [('words', eval_words[:30]), ('name', eval_names[:30])]:
        outs = generate(qwen, [e['instr'] for e in exs], max_new_tokens=300)
        q_res[kind] = sum(follows(e, o) for e, o in zip(exs, outs)) / len(exs)
    q_res['demo'] = generate(qwen, DEMO_PROMPTS, max_new_tokens=300)
    print(f"Qwen2.5-0.5B-Instruct (30 prompts each): words {q_res['words']:.0%} | name {q_res['name']:.0%}")
    print('\nPrompt   :', DEMO_PROMPTS[1])
    print('Generated:', q_res['demo'][1][:420])
    del qwen
    if device == 'mps': torch.mps.empty_cache()


The instruction-tuned 0.5B model is far better at following these instructions, because its post-training was much broader and it has about 12x more parameters. That capability gap is the ceiling for a 42M model. BabyQwen-3 stays with Baby Qwen and asks a narrower question: can a small LoRA adapter on top of `baby_qwen_sft` teach it **new facts**, and what are the limits?

---
## 10. Reload and Verify the Saved Models

A saved checkpoint is only useful if it reloads and behaves identically. We reload each artifact from disk and compare greedy outputs on the demo prompts with the in-memory results from above. For LoRA we also **merge** the adapter, which bakes $\Delta W = \frac{\alpha}{r}BA$ into the base weights so inference needs no PEFT.

In [ ]:
# 1) Full fine-tuned model: plain Hugging Face folder
reloaded = AutoModelForCausalLM.from_pretrained(SFT_DIR, dtype=torch.float32).to(device)
same_full = generate(reloaded, DEMO_PROMPTS) == results['full']['demo']
print(f'Reloaded {SFT_DIR}: outputs identical to in-memory model -> {same_full}')
del reloaded

# 2) LoRA: reload adapter on a fresh base, merge, and compare
base = load_base()
merged = PeftModel.from_pretrained(base, LORA_DIR).merge_and_unload()
same_lora = generate(merged, DEMO_PROMPTS) == results['lora']['demo']
n_merged = sum(p.numel() for p in merged.parameters())
print(f'Merged LoRA adapter into base ({n_merged:,} params, no adapter modules): outputs identical -> {same_lora}')
del merged, base

---
## 10. Optional Preview: Preference Tuning with DPO

SFT taught the model to *answer*. Preference tuning teaches it to prefer **better** answers over worse ones. Here we run the Week 5 recipe (see `5.2-LLM-Alignment-RLHF-DPO-and-GRPO`) on this SFT model:

$$\mathcal{L}_{\mathrm{DPO}}=-\,\mathbb{E}\Big[\log\sigma\Big(\beta\log\frac{\pi_\theta(y_w\mid x)}{\pi_{\mathrm{ref}}(y_w\mid x)}-\beta\log\frac{\pi_\theta(y_l\mid x)}{\pi_{\mathrm{ref}}(y_l\mid x)}\Big)\Big]$$

- **Policy** $\pi_\theta$: the SFT model being updated. **Reference** $\pi_{\mathrm{ref}}$: a frozen copy of the same SFT model.
- **Preference pairs, no human labels needed.** For each prompt we sample several answers from the SFT model and label them with the checkable rule from section 3: an answer that contains the requested word or name is $y_w$ (chosen), one that does not is $y_l$ (rejected).

> **Read this before running.** This section is a preview of the mechanics, and the result is likely to be **noisy**. A 42M model follows the instruction only part of the time, so many prompts yield no clean pair. The evaluation set is 60 prompts per task, so a difference of a few points between SFT and DPO is within sampling error, and DPO can also shift output length or style. Do not expect a clear win over SFT.
>
> **It cannot affect the next notebook.** The section is off by default (`RUN_DPO = False`). When enabled it writes only to `baby_qwen_dpo/` and never touches `baby_qwen_sft/`, which is what BabyQwen-3 loads. BabyQwen-3 therefore works the same whether or not you run this.

In [ ]:
RUN_DPO = False        # True: build preference pairs and train (about 10-20 min on a GPU or Apple Silicon, much longer on CPU)
DPO_DIR = 'baby_qwen_dpo'   # saved separately; BabyQwen-3 keeps using baby_qwen_sft

BETA = 0.1             # strength of the pull toward the reference model
N_PROMPTS = 300        # prompts to sample answers for
N_SAMPLES = 4          # answers sampled per prompt
MIN_PAIRS = 60         # below this, training is skipped (too little signal)
DPO_EPOCHS, DPO_LR, DPO_BS = 2, 1e-5, 2   # BS counts preference pairs (2 sequences each)

def seq_logps(m, ids, lab, att):
    # log pi(y | x): sum of response-token log-probs (prompt tokens are masked with -100)
    logits = m(input_ids=ids, attention_mask=att).logits[:, :-1].float()
    tgt = lab[:, 1:]
    tok = -F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=-100, reduction='none')
    return tok.view_as(tgt).sum(-1)

def dpo_loss(logp_w, logp_l, ref_w, ref_l, beta=0.1):
    # margin = beta * [(log pi_theta(y_w) - log pi_ref(y_w)) - (log pi_theta(y_l) - log pi_ref(y_l))]
    margin = beta * ((logp_w - ref_w) - (logp_l - ref_l))
    return -F.logsigmoid(margin).mean(), margin

# Sanity check: when the policy equals the reference every margin is 0, so the loss is log 2
zero = torch.zeros(4)
init_loss, _ = dpo_loss(zero, zero, zero, zero)
print(f'DPO loss at initialisation: {init_loss.item():.4f}  (log 2 = {math.log(2):.4f})')

In [ ]:
def load_sft():
    return AutoModelForCausalLM.from_pretrained(SFT_DIR, dtype=torch.float32).to(device)

@torch.no_grad()
def sample_responses(m, instructions, n, temperature=0.8, max_new_tokens=200, bs=10):
    # n sampled answers per instruction (temperature sampling, so the answers differ)
    m.eval()
    tokenizer.padding_side = 'left'
    outs = []
    for i in range(0, len(instructions), bs):
        batch = instructions[i:i + bs]
        enc = tokenizer([chat_prompt(x) for x in batch], return_tensors='pt', padding=True).to(m.device)
        g = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=0.95,
                       num_return_sequences=n, eos_token_id=[IM_END, EOT], pad_token_id=tokenizer.pad_token_id)
        texts = tokenizer.batch_decode(g[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        outs += [[t.strip() for t in texts[j * n:(j + 1) * n]] for j in range(len(batch))]
    tokenizer.padding_side = 'right'
    return outs

def build_pairs(m, examples, n_prompts, n_samples):
    # On-policy pairs: keep prompts where the model produced both a following and a non-following answer
    prompts = [e for e in examples if e['kind'] in ('words', 'name')][:n_prompts]
    samples = sample_responses(m, [e['instr'] for e in prompts], n_samples)
    pairs, rates = [], []
    for e, outs in zip(prompts, samples):
        ok = [o for o in outs if o and follows(e, o)]
        bad = [o for o in outs if not (o and follows(e, o))]
        rates.append(len(ok) / len(outs))
        if ok and bad:
            pairs.append(dict(instr=e['instr'], w=ok[0], l=bad[0]))
    return pairs, sum(rates) / max(1, len(rates)), len(prompts)

def dpo_train(policy, ref, pairs, epochs, lr, beta, bs):
    params = [p for p in policy.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, betas=(0.9, 0.95), weight_decay=0.0)
    log, step = [], 0
    for ep in range(epochs):
        order = list(range(len(pairs))); random.Random(ep).shuffle(order)
        for b in range(0, len(order), bs):
            batch = [pairs[j] for j in order[b:b + bs]]
            seqs = [s for p in batch for s in (dict(instr=p['instr'], resp=p['w']), dict(instr=p['instr'], resp=p['l']))]
            ids, lab, att = collate(seqs)                     # rows alternate: chosen, rejected, chosen, ...
            policy.train()
            lp = seq_logps(policy, ids, lab, att)
            with torch.no_grad():
                rf = seq_logps(ref, ids, lab, att)
            loss, margin = dpo_loss(lp[0::2], lp[1::2], rf[0::2], rf[1::2], beta)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); step += 1
            log.append((loss.item(), (margin > 0).float().mean().item()))
        recent = log[-max(1, math.ceil(len(pairs) / bs)):]
        print(f'[DPO] epoch {ep + 1}/{epochs} | loss {sum(l for l, _ in recent) / len(recent):.4f} | '
              f'preference accuracy {sum(a for _, a in recent) / len(recent):.0%}')
    return log

In [ ]:
if RUN_DPO:
    policy = load_sft()
    if 'full' not in results:       # e.g. the notebook was restarted: re-score the saved SFT model
        results['full'] = evaluate(policy, 'SFT (reloaded)')
    pairs, follow_rate, n_prompts = build_pairs(policy, train_examples, N_PROMPTS, N_SAMPLES)
    print(f'SFT model follows the instruction in {follow_rate:.0%} of sampled answers; '
          f'{len(pairs)} usable pairs from {n_prompts} prompts')
    if len(pairs) < MIN_PAIRS:
        print(f'Only {len(pairs)} pairs (< {MIN_PAIRS}): skipping DPO. Raise N_PROMPTS or N_SAMPLES, or use a stronger SFT model.')
    else:
        ref = load_sft().eval()
        for p in ref.parameters(): p.requires_grad_(False)
        dpo_log = dpo_train(policy, ref, pairs, DPO_EPOCHS, DPO_LR, BETA, DPO_BS)
        results['dpo'] = evaluate(policy, 'SFT + DPO')
        policy.save_pretrained(DPO_DIR); tokenizer.save_pretrained(DPO_DIR)
        print(f'Saved to {DPO_DIR}/ (baby_qwen_sft/ is unchanged, so BabyQwen-3 is unaffected)')
        del ref
    del policy
    if device == 'mps': torch.mps.empty_cache()
else:
    print('Skipped (RUN_DPO = False). The next notebook continues from baby_qwen_sft/ either way.')

In [ ]:
def ci95(p, n):
    return 1.96 * math.sqrt(p * (1 - p) / n)

if 'dpo' in results:
    for key, n in (('words', len(eval_words)), ('name', len(eval_names))):
        a, b = results['full'][key], results['dpo'][key]
        noise = math.sqrt(ci95(a, n) ** 2 + ci95(b, n) ** 2)
        print(f'{key:<6} SFT {a:.0%} +/-{ci95(a, n):.0%} | DPO {b:.0%} +/-{ci95(b, n):.0%} | '
              f'difference {b - a:+.0%} | outside sampling noise: {abs(b - a) > noise}')
    for lab, key in (('SFT', 'full'), ('DPO', 'dpo')):
        lens = [len(t.split()) for t in results[key]['demo']]
        print(f'{lab} mean demo length: {sum(lens) / len(lens):.0f} words')
else:
    print('No DPO results to compare (RUN_DPO is False or too few pairs).')

**How to read the comparison.** The `+/-` values are 95% sampling intervals for a rate measured on 60 prompts. Treat a difference as real only if the printout says it is outside sampling noise, and even then check the demo length: DPO often makes answers longer or shorter because the chosen answers differ in length from the rejected ones. The preference accuracy printed during training only says the policy ranks training pairs correctly, not that it generalizes.

**Why this is a preview.** Real preference tuning uses far stronger models and far more pairs. Here the signal is thin (few clean pairs) and the checker only measures one thing, so the useful lesson is the mechanics: a frozen reference, response-only log-probabilities, a loss of exactly $\log 2$ at the start, and a margin that grows as the policy moves.

---
## Summary & Key Takeaways

1. **SFT = chat template + response-only loss.** Masking prompt tokens with `-100` trains the model to *answer*, not to predict the question.
2. **Measure, don't eyeball.** Building instructions from text made the answer checkable, so instruction-following is a number (see the results table), not an impression.
3. **LoRA vs full fine-tuning is a trade-off.** LoRA trains a small fraction of the weights and saves a tiny adapter. Full fine-tuning updates everything and can adapt the embeddings. On a 42M model the whole model is cheap to train, so the usual LoRA advantage matters less than on a 7B model.
4. **SFT changes behavior, not capability.** Baby Qwen learned to follow a format and, to a degree, copy requested words and names. It did not learn new facts or reasoning. Pretraining scale sets the ceiling, which is the contrast with the 0.5B-Instruct model.
5. **Next: BabyQwen-3** loads `baby_qwen_sft` and trains a LoRA adapter that injects *new factual knowledge* (a made-up story-world), then examines where that succeeds and where the model still hallucinates.

6. **Optional: DPO preview.** Section 10 applies the Week 5 preference-tuning recipe to this SFT model. It is off by default, noisy at 42M parameters, and saved separately so the series continues from SFT.

**Artifacts saved here**: `baby_qwen_sft/` (full model) and `baby_qwen_lora_adapter/` (adapter), plus `baby_qwen_dpo/` only if you enable `RUN_DPO`.